# 牛津 Tutorial LLM 仿真 (v6.0) - AI商业模式类型学 + PRISMA

## Cell 1 - Persona Prompt (Oxford + HBS)

> 将以下 persona 注入任意 LLM (或静态仿真) 即可开启本单元的牛津 tutorial。本 notebook 用**静态 if/else 分支**模拟 Socratic 追问, 不真调 API (anti-stall)。

**Persona**:
You are an Oxford tutorial fellow in **AI商业模式类型学 + PRISMA系统文献综述**. You never give direct answers. You use Socratic questioning. You act as an HBS devil's advocate, challenging every vague claim about "AI business model". You reject ungrounded typology judgments. You end each turn with a probing question. If the student cannot defend a classification, you descend one scaffold level (worked -> faded -> independent) but never hand them the answer.

**禁忌**: 禁直接给答案; 禁替学生归类; 禁接受"它就是AI原生产品"这类无依据断言。每次回复必须以一个追问结尾。


## Cell 2 - Pre-tutorial Task (强制 retrieval)

> 开课前 24h 必须提交。这是 retrieval practice (提取练习) - 不提交不得进入 tutorial。

**提交物** (写进 `student_pre.py` 或 markdown):
1. **类型学预判**: 给定 5 个企业 (Perplexity / Hugging Face / Sierra / Cursor / Midjourney), 各归类到 AI 商业模式五大类型之一, 并用一句话写明"核心驱动力"依据。
2. **PRISMA 计划**: 你将如何用 `arxiv.Search(query=...)` + pandas 完成"识别->去重->筛选->综合"四步? 写出每步的 pandas 方法名。
3. **天道推演沙盘**: 推演"Agent经济主导"分支下, AI基础设施类企业的护城河会如何被侵蚀? (3 层推演: immediate -> near -> far)

> 提交后, tutor 会从你的预判中挑**最薄弱的一处**作为 tutorial 起点 (不是最熟的)。


In [ ]:
# Cell 3 - Multi-turn Socratic Loop (静态 if/else 仿真, >=4 轮, >=5 苏格拉底问)
# 不调 openai/anthropic API。用学生提交的关键词匹配, 模拟 Socratic 追问链。

STUDENT_SUBMISSION = {
    "perplexity": "AI原生产品",        # 学生可能答对
    "huggingface": "AI平台",            # 对
    "sierra": "Agent经济",              # 对
    "cursor": "AI增强产品",             # 错! Cursor 是 AI原生 (剥离AI产品不成立)
    "midjourney": "AI基础设施",          # 错! 应为 AI原生产品
    "dedup_method": "drop_duplicates",  # 对
    "prisma_filter": "year>2020",       # 不完整 (缺相关性筛选)
}

# 苏格拉底问库 (>=5)
SOCRATIC_QUESTIONS = [
    "Q1(为什么): 你说 Cursor 是 AI增强产品 - 那么剥离 AI 后, Cursor 这个产品还成立吗? 凭什么?",
    "Q2(反例): 若 Midjourney 是 AI基础设施, 那 OpenAI (卖 API) 算什么? 二者的收入模型差异是什么?",
    "Q3(若前提变): 假设 Hugging Face 明天关闭模型托管, 只留榜单, 它还是 AI平台 吗? 平台护城河的根基变了吗?",
    "Q4(凭什么): 你用 'year>2020' 筛选 - 凭什么认为 2020 前的 AI商业模式文献无关? 这会引入什么偏差?",
    "Q5(如何): 如何用 pandas 同时按 title 去重 AND 按相关性筛选? 给出布尔索引的写法。",
    "Q6(反例): 你把 Sierra 归为 Agent经济 - 若 Sierra 的 outcome-based 定价 90% 失败转回 SaaS, 它退化成哪一类?",
]

def socratic_round(student_ans, round_num):
    """静态 if/else 模拟 Socratic 追问 - 按 round_num 推进脚手架渐退"""
    print(f"\n===== Round {round_num} =====")
    if round_num == 1:
        # 从最薄弱处切入: Cursor 误判
        print("Tutor: 你的 Cursor=AI增强产品 判断, 让我们检验。")
        print(SOCRATIC_QUESTIONS[0])  # 为什么
        if student_ans.get("cursor") == "AI增强产品":
            print(">> 学生需 defending: 剥离 AI 后 Cursor 是否成立? 若答不出 -> 降一级 scaffold (回退 worked example)")
            return "descend_to_worked"
    if round_num == 2:
        print("Tutor: 上轮你未能 defense。给个 worked example: 'OpenAI GPT API 按 token 计费 -> 核心驱动力=算力+模型 -> AI基础设施'。")
        print(SOCRATIC_QUESTIONS[1])  # 反例
        return "faded"
    if round_num == 3:
        print("Tutor: 现在用相同链条判断 Cursor。填空: Cursor 的核心驱动力=? 收入模型=? -> 类型=?")
        print(SOCRATIC_QUESTIONS[5])  # 如何 (Agent经济退化反例)
        return "independent_probe"
    if round_num == 4:
        print("Tutor: 转到 PRISMA。你的筛选条件 'year>2020' 不完整。")
        print(SOCRATIC_QUESTIONS[3])  # 凭什么
        print(SOCRATIC_QUESTIONS[4])  # 如何
        return "prisma_defense"
    return "exit"

# 运行 >=4 轮 Socratic loop
for r in range(1, 5):
    state = socratic_round(STUDENT_SUBMISSION, r)
    print(f">> scaffold state: {state}")
print("\n[Tutorial 结束 - 学生应已自纠 Cursor/Midjourney 误判 + 补全 PRISMA 筛选条件]")


In [ ]:
# Cell 4 - student_model.json 读写 (跨单元复用, 记录掌握度/盲点)
import json, os

student_model = {
    "student_id": "phd_s4_d1",
    "unit": "skill-4-day-1",
    "mastery": {
        "S1_类型识别": {"Cursor": 0.2, "Midjourney": 0.3, "Perplexity": 0.9, "Sierra": 0.8},
        "S2_PRISMA流程": {"去重": 0.9, "筛选条件完整性": 0.4, "流程图": 0.6},
        "S3_天道推演": {"沙盘分支": 0.5}
    },
    "blind_spots": [
        "混淆 AI增强产品 vs AI原生产品 (剥离AI测试未内化)",
        "PRISMA 筛选条件不完整 (缺相关性维度)",
        "Midjourney 误归为基础设施 (未用收入模型判据)"
    ],
    "scaffold_level": "faded",   # worked -> faded -> independent
    "next_review_units": ["skill-4-day-2 (价值创造+定价)", "module-r-day-r4 (PRISMA系统综述)"],
    "daily_limit": {"tutorial_used_today": 1, "max_per_day": 1}
}

path = os.path.join(os.path.dirname(os.path.abspath(".")) if False else ".", "student_model.json")
with open("student_model.json", "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)
print("WROTE student_model.json")
print(json.dumps(student_model, ensure_ascii=False, indent=2))


## Cell 5 - Hattie 4 级 Formative Feedback

> Hattie (2007) 4 级反馈。避免 Self 级表扬 (无效), 聚焦 Task/Process/Self-Reg/Feed-Forward。

**[TASK] 任务级** (这个答案对不对):
- Cursor 归类错误。AI增强产品的判据是"剥离AI后产品仍成立" (如 Salesforce Einstein 离开 AI 仍是 CRM)。Cursor 剥离 AI 后只剩文本编辑器骨架, 不成立 -> 应归 AI原生产品。
- Midjourney 归类错误。AI基础设施按用量计费卖算力 (OpenAI API)。Midjourney 卖的是生成能力本身 (订阅+用量混合) -> AI原生产品, 非基础设施。

**[PROCESS] 过程级** (你用的策略对不对):
- 你的归类策略缺"剥离AI测试"这一步, 直接看表面"有AI"就归增强产品。正确流程: ① 看核心驱动力 ② 验收入模型 ③ 做"剥离AI"思想实验 ④ 归类。
- PRISMA 筛选用单一 `year>2020` 维度。应多维度布尔索引: `(df.year>=2020) & (df.relevance_score>=threshold) & (~df.duplicated)`。

**[SELF-REG] 自我调节级** (你如何监控自己的理解):
- 在归类前, 你是否自问"凭什么"? 若没有, 这是元认知缺口。下次每个归类强制写"依据:"一行, 依据不能为空。
- PRISMA 筛选条件写完后, 反向问"这个条件会排除哪些不该排除的?" - 这是 self-reg 的反向检验。

**[FEED-FORWARD] 前馈级** (下一步怎么做):
- 重做 practice.md 的 D1 Faded 阶段 (5 篇 arxiv 摘要独立归类), 直到 5/5 全对。
- 补全 PRISMA 筛选为多维度布尔索引, 在 starter.ipynb TODO4 处实现, 跑出 160->96->30->30 真实数字。
- 下一单元 (Day 2 价值创造+定价) 会用今天的类型学, 请先复习五大类型的"收入模型"列。


## Cell 6 - 限频 + Exit Artifact

**限频 (防依赖)**: 本单元 tutorial 每天**最多 1 次**。Oxford tutorial 的价值在于"逼你先想", 而非"替你想"。已用 1 次 (`student_model.daily_limit.tutorial_used_today=1`), 今日不再开。明日重置。

> 设计依据: Vygotsky 最近发展区 + 防止 LLM 依赖 (arxiv 2024-2025 Socratic LLM 论文共识 - 限频 + 渐退脚手架)。

**Exit Artifact (tutorial 结束必须产出)**:
1. **2-3 个盲点** (从 student_model.blind_spots 中确认你认同的):
   - 盲点1: ___________________________________
   - 盲点2: ___________________________________
   - 盲点3: ___________________________________
2. **推荐复习单元** (从 next_review_units 选 1 个, 写复习计划):
   - 我将复习 _________________, 用 ____(时间) 做 ____(具体动作)。
3. **一句话承诺**: 下次归类前, 我会先做 ____________ 测试再下判断。

> 提交 exit artifact 后, tutor 不再回复 - 剩余靠 deliberate practice (practice.md) + spaced retrieval (schedule.json) 内化。
